<a href="https://colab.research.google.com/github/ai1108/kebbi-timebox-love-story/blob/main/kebbi_timebox_flex_with_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Colab 左邊點 🔑 Secrets，新增以下三組密鑰（貼上你自己的值，不要寫在這裡）：

LINE_CHANNEL_ACCESS_TOKEN
LINE_GROUP_ID
NGROK_AUTHTOKEN

In [ ]:
!pip -q install paho-mqtt requests

In [ ]:
import uuid
import time
import requests
import paho.mqtt.client as mqtt

from google.colab import userdata


# ========================================
# LINE 設定
# ========================================

LINE_TOKEN = userdata.get("LINE_CHANNEL_ACCESS_TOKEN")
LINE_GROUP_ID = userdata.get("LINE_GROUP_ID")


# ========================================
# MQTT 設定
# ========================================

MQTT_BROKER = "broker.emqx.io"
MQTT_PORT = 1883

MQTT_TOPIC = "kebbi_timebox_83026"


# ========================================
# 傳送訊息到 LINE
# ========================================

def send_to_line(text):

    url = "https://api.line.me/v2/bot/message/push"

    headers = {
        "Authorization": f"Bearer {LINE_TOKEN}",
        "Content-Type": "application/json"
    }

    data = {
        "to": LINE_GROUP_ID,
        "messages": [
            {
                "type": "text",
                "text": text
            }
        ]
    }

    try:

        response = requests.post(
            url,
            headers=headers,
            json=data,
            timeout=15
        )

        if response.status_code == 200:

            print("✅ LINE 發送成功！")

        else:

            print("❌ LINE 發送失敗")
            print("HTTP =", response.status_code)
            print(response.text)

    except Exception as e:

        print("❌ LINE 發送發生錯誤：")
        print(e)


# ========================================
# MQTT 連線成功時
# ========================================

def on_connect(client, userdata, flags, reason_code, properties):

    print("MQTT 回傳 =", reason_code)

    if reason_code == 0:

        print("✅ MQTT 連線成功")

        client.subscribe(MQTT_TOPIC)

        print("📡 正在等待凱比...")
        print("Topic =", MQTT_TOPIC)

    else:

        print("❌ MQTT 連線失敗")


# ========================================
# 收到凱比資料
# ========================================

def on_message(client, userdata, msg):

    try:

        text = msg.payload.decode("utf-8")

        print("")
        print("==============================")
        print("🤖 收到凱比資料！")
        print("Topic：", msg.topic)
        print("")
        print(text)
        print("==============================")
        print("")

        # 收到凱比資料後直接傳 LINE
        send_to_line(text)

    except Exception as e:

        print("❌ MQTT 訊息處理失敗")
        print(e)


# ========================================
# 建立 MQTT Client
# ========================================

client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION2,
    client_id="kebbi_colab_" + uuid.uuid4().hex[:8]
)

client.on_connect = on_connect
client.on_message = on_message


# ========================================
# 連接 MQTT
# ========================================

print("🔄 正在連接 MQTT...")

client.connect(
    MQTT_BROKER,
    MQTT_PORT,
    keepalive=60
)

client.loop_start()

time.sleep(3)

 **沒有凱比時怎麼測**

In [ ]:
test_memory = """【時光寶盒－爺奶戀愛史】
第一次約會：我們第一次約會是在西門町看電影。
看電影趣事：阿嬤看到恐怖片的時候嚇了一大跳，還把爆米花打翻了。"""

info = client.publish(
    "kebbi_timebox_83026",
    test_memory
)

info.wait_for_publish()

print("測試訊息發送結果 =", info.rc)

**真的拿到凱比**

In [ ]:
摸凱比頭
↓
凱比打招呼
↓
問第一次約會
↓
語音辨識
↓
answer_1
↓
問電影趣事
↓
語音辨識
↓
answer_2
↓
組合 memory_text
↓
CodeLab：

發送消息 memory_text
至 Topic kebbi_timebox_83026

↓
EMQX
↓
Colab 自動收到
↓
send_to_line()
↓
LINE 家庭群組

圖文推播從這裡開始跑 圖片檔案名要記得改成timebox

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving timebox.png to timebox.png


In [ ]:
!pip -q install flask pyngrok paho-mqtt requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 4.8 MB/s eta 0:00:00


In [ ]:
import threading
import time

from flask import Flask, send_from_directory
from pyngrok import ngrok
from google.colab import userdata


IMAGE_FILENAME = "timebox.png"

app = Flask(__name__)


@app.route("/timebox-image")
def timebox_image():
    return send_from_directory(
        "/content",
        IMAGE_FILENAME
    )


def run_image_server():
    app.run(
        host="0.0.0.0",
        port=5000,
        use_reloader=False
    )


# 啟動 Flask
threading.Thread(
    target=run_image_server,
    daemon=True
).start()

time.sleep(2)


# 啟動 ngrok
NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")

ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(5000).public_url

IMAGE_URL = public_url + "/timebox-image"

print("✅ 圖片網址：")
print(IMAGE_URL)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


✅ 圖片網址：
https://d5a9-34-16-180-92.ngrok-free.app/timebox-image


In [ ]:
import requests
from google.colab import userdata


LINE_TOKEN = userdata.get("LINE_CHANNEL_ACCESS_TOKEN")
LINE_GROUP_ID = userdata.get("LINE_GROUP_ID")


def send_flex_to_line(memory_text):

    url = "https://api.line.me/v2/bot/message/push"

    headers = {
        "Authorization": f"Bearer {LINE_TOKEN}",
        "Content-Type": "application/json"
    }

    flex_message = {
        "type": "flex",
        "altText": "新的時光寶盒回憶",
        "contents": {
            "type": "bubble",

            # 上面的圖片
            "hero": {
                "type": "image",
                "url": IMAGE_URL,
                "size": "full",
                "aspectRatio": "1:1",
                "aspectMode": "cover"
            },

            # 下面的文字
            "body": {
                "type": "box",
                "layout": "vertical",
                "spacing": "md",
                "contents": [
                    {
                        "type": "text",
                        "text": "🕰️ 時光寶盒",
                        "weight": "bold",
                        "size": "xl"
                    },
                    {
                        "type": "text",
                        "text": "爺奶戀愛史",
                        "weight": "bold",
                        "size": "md"
                    },
                    {
                        "type": "separator",
                        "margin": "md"
                    },
                    {
                        "type": "text",
                        "text": memory_text,
                        "wrap": True,
                        "size": "sm",
                        "margin": "md"
                    }
                ]
            }
        }
    }

    payload = {
        "to": LINE_GROUP_ID,
        "messages": [
            flex_message
        ]
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=15
    )

    if response.status_code == 200:
        print("✅ LINE 圖文訊息發送成功！")
    else:
        print("❌ LINE 發送失敗")
        print("HTTP =", response.status_code)
        print(response.text)

In [ ]:
test_memory = """【時光寶盒－爺奶戀愛史】

第一次約會：
阿公和阿嬤第一次約會是在西門町看電影。

看電影趣事：
看到恐怖片時阿嬤嚇了一跳，還把爆米花打翻了。"""

send_flex_to_line(test_memory)

✅ LINE 圖文訊息發送成功！


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'ChatGPT Image 2026年9月1日 下午12_37_50.png', 'sample_data']
